In [1]:
import json
import glob
import os
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.storage.docstore.simple_docstore import SimpleDocumentStore
from llama_index.core.schema import TextNode
from llama_index.core.storage.storage_context import StorageContext

In [4]:
import json

with open("textbook_chunks_temp.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Number of entries: {len(data)}")
for i, entry in enumerate(data[:5]): 
    print(f"Entry #{i+1}: {entry}")

Number of entries: 99
Entry #1: {'id': 'chunk_002', 'chunk': 'This section provides a casual overview of Review of the latest sci-fi movie, highlighting basic steps and personal experiences. Many enthusiasts share their own tips and tricks, reflecting the subjective nature of this topic. Readers interested in Review of the latest sci-fi movie will find a range of opinions and methodologies, often influenced by cultural trends. Unlike academic subjects, this content focuses on practical enjoyment and personal preference. The tone is conversational, aiming to engage rather than to educate in a formal sense. Despite the informal nature, there are insights to gain for anyone curious about Review of the latest sci-fi movie.'}
Entry #2: {'id': 'chunk_003', 'chunk': 'The concept of Pythagorean theorem is fundamental in its respective field, underpinning many advanced theories and applications. Researchers have studied this topic extensively, exploring its mechanisms and implications across va

In [5]:
JSON_CHUNKS_DIR = "."
INDEX_DIR       = "/work/10405/ananyagm/ls6/hf_cache/bm25_index_storage"
TOP_K           = 5

In [6]:
def load_chunks(json_dir):
    docs = []
    for json_path in glob.glob(os.path.join(json_dir, "*.json")):
        with open(json_path, "r", encoding="utf-8") as f:
            raw = json.load(f)
        for entry in raw:
            chunk_text = entry.get("chunk", "").strip()
            if chunk_text:  
                docs.append(TextNode(text=chunk_text, id_=entry.get("id")))
    return docs

In [9]:
if os.path.isdir(INDEX_DIR) and os.listdir(INDEX_DIR):
    print("Loading existing BM25 index from disk...")
    storage_context = StorageContext.from_defaults(persist_dir=INDEX_DIR)
    docstore = SimpleDocumentStore.from_persist_path(INDEX_DIR)
else:
    print("No existing index found. Building BM25 index...")
    documents = load_chunks(JSON_CHUNKS_DIR)
    print(f"Loaded {len(documents)} non-empty documents.")
    
    docstore = SimpleDocumentStore()
    docstore.add_documents(documents) 
    
    os.makedirs(INDEX_DIR, exist_ok=True)
    docstore.persist()

No existing index found. Building BM25 index...
Loaded 99 non-empty documents.


In [10]:
print(f"Number of documents in docstore: {len(docstore.docs)}")
for node_id, node in docstore.docs.items():
    print(f"Doc ID: {node_id}, Text Length: {len(node.text)}")

Number of documents in docstore: 99
Doc ID: chunk_002, Text Length: 666
Doc ID: chunk_003, Text Length: 910
Doc ID: chunk_004, Text Length: 805
Doc ID: chunk_005, Text Length: 904
Doc ID: chunk_006, Text Length: 811
Doc ID: chunk_007, Text Length: 762
Doc ID: chunk_008, Text Length: 817
Doc ID: chunk_009, Text Length: 898
Doc ID: chunk_010, Text Length: 826
Doc ID: chunk_011, Text Length: 892
Doc ID: chunk_012, Text Length: 793
Doc ID: chunk_013, Text Length: 898
Doc ID: chunk_014, Text Length: 796
Doc ID: chunk_015, Text Length: 744
Doc ID: chunk_016, Text Length: 808
Doc ID: chunk_017, Text Length: 747
Doc ID: chunk_018, Text Length: 805
Doc ID: chunk_019, Text Length: 901
Doc ID: chunk_020, Text Length: 799
Doc ID: chunk_021, Text Length: 895
Doc ID: chunk_022, Text Length: 666
Doc ID: chunk_023, Text Length: 910
Doc ID: chunk_024, Text Length: 805
Doc ID: chunk_025, Text Length: 904
Doc ID: chunk_026, Text Length: 811
Doc ID: chunk_027, Text Length: 762
Doc ID: chunk_028, Text Leng

In [11]:
bm25_retriever = BM25Retriever.from_defaults(
    docstore=docstore,
    similarity_top_k=TOP_K,
)

In [12]:
# Re-ranker (R3)
def keyword_reranker(query, nodes):
    ranked = sorted(
        nodes,
        key=lambda node: sum(1 for word in query.split() if word.lower() in node.text.lower()),
        reverse=True
    )
    return ranked

# Retriever
def retrieve(question: str):
    if not question.strip():
        return []
    hits = bm25_retriever.retrieve(question)  # retrieve from BM25
    reranked_hits = keyword_reranker(question, hits)  # rerank using keyword overlap
    return [doc.text for doc in reranked_hits]  # return only text


In [14]:
if __name__ == "__main__":
    q = "What happened in French Revolution?"
    print(f"\n>>> Query: {q}")
    for i, chunk in enumerate(retrieve(q), 1):
        print(f"\nPassage #{i}\n{chunk}")


>>> Query: What happened in French Revolution?

=== Passage #1 ===
The concept of French Revolution is fundamental in its respective field, underpinning many advanced theories and applications. Researchers have studied this topic extensively, exploring its mechanisms and implications across various contexts. In practical terms, French Revolution can be observed in everyday phenomena, from simple experiments to real-world systems. Advanced understanding of this topic enables innovations and improvements in technology, education, and industry. Educational curricula often include comprehensive modules on this topic, ensuring students gain a solid theoretical and practical foundation. Overall, French Revolution remains a vibrant area of study, with ongoing research pushing the boundaries of what we know. This narrative paragraph is crafted to provide depth and fill out the required length for demonstration. It adds context and elaborates on nuances for clarity.
